In [0]:
"""
Databricks PySpark Pipeline
Silver Layer → Power BI-Ready Aggregations → Azure SQL + Gold Layer (ADLS Parquet)

Tables produced (12 total):
  NEW (chart-ready):
    1.  patient_age_group_summary
    2.  age_group_condition_summary
    3.  insurance_avg_billing_summary
    4.  gender_condition_summary
    5.  blood_type_condition_summary
    6.  condition_billing_summary
  EXISTING (KPI tables):
    7.  patient_demographics
    8.  medical_conditions_summary
    9.  hospital_performance_metrics
   10.  doctor_performance_summary
   11.  admission_type_analysis
   12.  insurance_provider_analysis

Each table is written to:
  → Azure SQL Database  (for Power BI DirectQuery / Import)
  → ADLS Gold Layer     (Parquet, partitioned — for downstream analytics)
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, avg, min, max,
    sum as spark_sum,
    round as spark_round,
    when, datediff, current_timestamp, desc
)
import logging


In [0]:
spark = SparkSession.builder \
    .appName("SilverToSQLAndGold_PowerBI") \
    .getOrCreate()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [0]:
# ── Azure Storage ──────────────────────────────────────────────────────────────
STORAGE_ACCOUNT  = "adlstoragesiddharth"
STORAGE_KEY      = dbutils.secrets.get(scope="DBSecrets", key="adls")
CONTAINER_SILVER = "silver"
CONTAINER_GOLD   = "gold"

ADLS_SILVER_PATH = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/healthcare/patient_data"
ADLS_GOLD_BASE   = f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/healthcare"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

# ── Azure SQL ──────────────────────────────────────────────────────────────────
SQL_SERVER   = "sqldbc37.database.windows.net"
SQL_DATABASE = "AzureSQLDB"
SQL_USERNAME = "siddharthdb"
SQL_PASSWORD = dbutils.secrets.get(scope="DBSecrets", key="azsqldb")
SQL_URL      = (
    f"jdbc:sqlserver://{SQL_SERVER}:1433;"
    f"database={SQL_DATABASE};"
    "encrypt=true;trustServerCertificate=false;loginTimeout=30;"
)

logger.info(f"Silver : {ADLS_SILVER_PATH}")
logger.info(f"Gold   : {ADLS_GOLD_BASE}")
logger.info(f"SQL    : {SQL_SERVER} / {SQL_DATABASE}")


In [0]:
# ============================================================
# WRITER HELPERS
# Two destinations: Azure SQL  +  ADLS Gold (Parquet)
# ============================================================

def write_to_sql(df, table_name, mode="overwrite"):
    """Write a DataFrame to Azure SQL Database via JDBC."""
    try:
        df.write \
            .format("jdbc") \
            .mode(mode) \
            .option("url",      SQL_URL) \
            .option("dbtable",  table_name) \
            .option("user",     SQL_USERNAME) \
            .option("password", SQL_PASSWORD) \
            .option("driver",   "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
            .save()
        logger.info(f"  [SQL  OK] {table_name}")
    except Exception as e:
        logger.error(f"  [SQL FAIL] {table_name}: {e}")
        raise


def write_to_gold(df, folder_name, partition_by=None, mode="overwrite"):
    """Write a DataFrame to the Gold layer on ADLS as Parquet."""
    path = f"{ADLS_GOLD_BASE}/{folder_name}"
    try:
        writer = df.write.format("parquet").mode(mode)
        if partition_by:
            writer = writer.partitionBy(partition_by)
        writer.save(path)
        logger.info(f"  [GOLD OK] {path}")
    except Exception as e:
        logger.error(f"  [GOLD FAIL] {path}: {e}")
        raise


def write_both(df, table_name, folder_name, partition_by=None):
    """Write to SQL and Gold in one call."""
    write_to_sql(df, table_name)
    write_to_gold(df, folder_name, partition_by)


In [0]:
# ============================================================
# READ FROM SILVER
# ============================================================
logger.info("Reading Silver layer...")
df_silver = spark.read.format("parquet").load(ADLS_SILVER_PATH)
logger.info(f"Record count: {df_silver.count():,}")

# Age group derived column (reused across aggregations)
df = df_silver.withColumn(
    "age_group",
    when(col("Age") < 18,  "Child")
   .when(col("Age") < 60,  "Adult")
   .otherwise("Senior")
)


In [0]:
# ============================================================
# TABLE 1 : patient_age_group_summary
# Chart    : Hospital Usage by Age Group (bar)
# ============================================================
df_age_group = (
    df
    .groupBy("age_group")
    .agg(
        count("*")                                   .alias("patient_count"),
        spark_round(avg("Age"), 2)                   .alias("avg_age"),
        min("Age")                                   .alias("min_age"),
        max("Age")                                   .alias("max_age"),
        spark_round(avg("Billing Amount"), 2)        .alias("avg_billing_amount"),
        spark_round(spark_sum("Billing Amount"), 2)  .alias("total_billing_amount")
    )
    .orderBy(desc("patient_count"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 1] {df_age_group.count()} rows")
df_age_group.show(truncate=False)

write_both(df_age_group,
           table_name  = "patient_age_group_summary",
           folder_name = "patient_age_group_summary")


In [0]:
# ============================================================
# TABLE 2 : age_group_condition_summary
# Chart    : Medical Conditions Across Age Groups (stacked bar)
# ============================================================
df_age_condition = (
    df
    .groupBy("age_group", "Medical Condition")
    .agg(
        count("*")                             .alias("patient_count"),
        spark_round(avg("Billing Amount"), 2)  .alias("avg_billing_amount")
    )
    .withColumnRenamed("Medical Condition", "medical_condition")
    .orderBy("age_group", desc("patient_count"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 2] {df_age_condition.count()} rows")
df_age_condition.show(20, truncate=False)

write_both(df_age_condition,
           table_name   = "age_group_condition_summary",
           folder_name  = "age_group_condition_summary",
           partition_by = "age_group")


In [0]:
# ============================================================
# TABLE 3 : insurance_avg_billing_summary
# Chart    : Average Billing by Insurance Provider (bar)
# ============================================================
df_insurance_billing = (
    df
    .groupBy("Insurance Provider")
    .agg(
        count("*")                                   .alias("covered_patients"),
        spark_round(avg("Billing Amount"), 2)        .alias("avg_billing_amount"),
        spark_round(spark_sum("Billing Amount"), 2)  .alias("total_billing_amount"),
        spark_round(min("Billing Amount"), 2)        .alias("min_billing_amount"),
        spark_round(max("Billing Amount"), 2)        .alias("max_billing_amount")
    )
    .withColumnRenamed("Insurance Provider", "insurance_provider")
    .orderBy(desc("avg_billing_amount"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 3] {df_insurance_billing.count()} rows")
df_insurance_billing.show(truncate=False)

write_both(df_insurance_billing,
           table_name  = "insurance_avg_billing_summary",
           folder_name = "insurance_avg_billing_summary")


In [0]:
# ============================================================
# TABLE 4 : gender_condition_summary
# Chart    : Medical Condition by Gender (stacked bar)
# ============================================================
df_gender_condition = (
    df
    .groupBy("Gender", "Medical Condition")
    .agg(
        count("*")                             .alias("patient_count"),
        spark_round(avg("Age"), 2)             .alias("avg_age"),
        spark_round(avg("Billing Amount"), 2)  .alias("avg_billing_amount")
    )
    .withColumnRenamed("Gender",            "gender")
    .withColumnRenamed("Medical Condition", "medical_condition")
    .orderBy("gender", desc("patient_count"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 4] {df_gender_condition.count()} rows")
df_gender_condition.show(20, truncate=False)

write_both(df_gender_condition,
           table_name   = "gender_condition_summary",
           folder_name  = "gender_condition_summary",
           partition_by = "gender")


In [0]:
# ============================================================
# TABLE 5 : blood_type_condition_summary
# Chart    : Blood Type vs Medical Condition (stacked bar)
# ============================================================
df_blood_condition = (
    df
    .groupBy("Blood Type", "Medical Condition")
    .agg(
        count("*")                             .alias("patient_count"),
        spark_round(avg("Billing Amount"), 2)  .alias("avg_billing_amount")
    )
    .withColumnRenamed("Blood Type",        "blood_type")
    .withColumnRenamed("Medical Condition", "medical_condition")
    .orderBy("blood_type", desc("patient_count"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 5] {df_blood_condition.count()} rows")
df_blood_condition.show(30, truncate=False)

write_both(df_blood_condition,
           table_name   = "blood_type_condition_summary",
           folder_name  = "blood_type_condition_summary",
           partition_by = "blood_type")


In [0]:
# ============================================================
# TABLE 6 : condition_billing_summary
# Chart    : Average Billing by Medical Condition (bar)
# ============================================================
df_condition_billing = (
    df
    .groupBy("Medical Condition")
    .agg(
        count("*")                                   .alias("patient_count"),
        spark_round(avg("Billing Amount"), 2)        .alias("avg_billing_amount"),
        spark_round(spark_sum("Billing Amount"), 2)  .alias("total_billing_amount"),
        spark_round(min("Billing Amount"), 2)        .alias("min_billing_amount"),
        spark_round(max("Billing Amount"), 2)        .alias("max_billing_amount"),
        spark_round(avg("Age"), 2)                   .alias("avg_age")
    )
    .withColumnRenamed("Medical Condition", "medical_condition")
    .orderBy(desc("avg_billing_amount"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 6] {df_condition_billing.count()} rows")
df_condition_billing.show(truncate=False)

write_both(df_condition_billing,
           table_name  = "condition_billing_summary",
           folder_name = "condition_billing_summary")


In [0]:
# ============================================================
# TABLE 7 : patient_demographics  (dimension table)
# ============================================================
df_patient_demographics = (
    df_silver
    .select(
        col("Name")               .alias("patient_name"),
        col("Age")                .alias("age"),
        col("Gender")             .alias("gender"),
        col("Blood Type")         .alias("blood_type"),
        col("Insurance Provider") .alias("insurance_provider"),
        col("Date of Admission")  .alias("admission_date"),
        col("Discharge Date")     .alias("discharge_date"),
        col("Medical Condition")  .alias("medical_condition"),
        col("Admission Type")     .alias("admission_type"),
        col("Hospital")           .alias("hospital"),
        col("Doctor")             .alias("doctor"),
        col("Billing Amount")     .alias("billing_amount"),
        col("Room Number")        .alias("room_number"),
        col("Medication")         .alias("medication"),
        col("Test Results")       .alias("test_results")
    )
    .dropDuplicates(["patient_name", "admission_date"])
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 7] {df_patient_demographics.count():,} rows")

write_both(df_patient_demographics,
           table_name   = "patient_demographics",
           folder_name  = "patient_demographics",
           partition_by = "gender")


In [0]:
# ============================================================
# TABLE 8 : medical_conditions_summary  (condition-level KPIs)
# ============================================================
df_conditions = (
    df_silver
    .groupBy("Medical Condition")
    .agg(
        count("*")                                   .alias("patient_count"),
        spark_round(avg("Age"), 2)                   .alias("avg_age"),
        min("Age")                                   .alias("min_age"),
        max("Age")                                   .alias("max_age"),
        spark_round(avg("Billing Amount"), 2)        .alias("avg_billing_amount"),
        spark_round(max("Billing Amount"), 2)        .alias("max_billing_amount"),
        spark_round(min("Billing Amount"), 2)        .alias("min_billing_amount"),
        spark_round(spark_sum("Billing Amount"), 2)  .alias("total_billing_amount")
    )
    .withColumnRenamed("Medical Condition", "condition_name")
    .orderBy(desc("patient_count"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 8] {df_conditions.count()} rows")
df_conditions.show(truncate=False)

write_both(df_conditions,
           table_name  = "medical_conditions_summary",
           folder_name = "medical_conditions_summary")


In [0]:
# ============================================================
# TABLE 9 : hospital_performance_metrics
# ============================================================
df_hospital_metrics = (
    df_silver
    .groupBy("Hospital")
    .agg(
        count("*")                                   .alias("total_patients"),
        spark_round(avg("Age"), 2)                   .alias("avg_patient_age"),
        spark_round(avg("Billing Amount"), 2)        .alias("avg_billing_amount"),
        spark_round(spark_sum("Billing Amount"), 2)  .alias("total_revenue"),
        spark_round(
            count(when(col("Test Results") == "Normal", 1)) / count("*") * 100, 2
        )                                            .alias("normal_test_pct"),
        count(when(col("Admission Type") == "Emergency", 1)) .alias("emergency_admissions"),
        count(when(col("Admission Type") == "Urgent",    1)) .alias("urgent_admissions"),
        count(when(col("Admission Type") == "Elective",  1)) .alias("elective_admissions")
    )
    .withColumnRenamed("Hospital", "hospital_name")
    .orderBy(desc("total_patients"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 9] {df_hospital_metrics.count():,} rows")
df_hospital_metrics.show(5, truncate=False)

write_both(df_hospital_metrics,
           table_name  = "hospital_performance_metrics",
           folder_name = "hospital_performance_metrics")


In [0]:
# ============================================================
# TABLE 10 : doctor_performance_summary
# ============================================================
df_doctor_metrics = (
    df_silver
    .groupBy("Doctor")
    .agg(
        count("*")                                   .alias("patients_treated"),
        spark_round(avg("Age"), 2)                   .alias("avg_patient_age"),
        spark_round(avg("Billing Amount"), 2)        .alias("avg_billing_per_patient"),
        spark_round(spark_sum("Billing Amount"), 2)  .alias("total_revenue"),
        count(when(col("Test Results") == "Normal",       1)) .alias("normal_count"),
        count(when(col("Test Results") == "Abnormal",     1)) .alias("abnormal_count"),
        count(when(col("Test Results") == "Inconclusive", 1)) .alias("inconclusive_count")
    )
    .withColumnRenamed("Doctor", "doctor_name")
    .orderBy(desc("patients_treated"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 10] {df_doctor_metrics.count():,} rows")
df_doctor_metrics.show(5, truncate=False)

write_both(df_doctor_metrics,
           table_name  = "doctor_performance_summary",
           folder_name = "doctor_performance_summary")


In [0]:
# ============================================================
# TABLE 11 : admission_type_analysis
# ============================================================
df_admission_analysis = (
    df_silver
    .groupBy("Admission Type")
    .agg(
        count("*")                                   .alias("total_admissions"),
        spark_round(avg("Age"), 2)                   .alias("avg_patient_age"),
        spark_round(avg("Billing Amount"), 2)        .alias("avg_cost"),
        spark_round(spark_sum("Billing Amount"), 2)  .alias("total_cost"),
        spark_round(
            avg(datediff(col("Discharge Date"), col("Date of Admission"))), 2
        )                                            .alias("avg_length_of_stay_days")
    )
    .withColumnRenamed("Admission Type", "admission_type")
    .orderBy(desc("total_admissions"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 11] {df_admission_analysis.count()} rows")
df_admission_analysis.show(truncate=False)

write_both(df_admission_analysis,
           table_name   = "admission_type_analysis",
           folder_name  = "admission_type_analysis",
           partition_by = "admission_type")


In [0]:
# ============================================================
# TABLE 12 : insurance_provider_analysis
# ============================================================
df_insurance_analysis = (
    df_silver
    .groupBy("Insurance Provider")
    .agg(
        count("*")                                   .alias("covered_patients"),
        spark_round(avg("Billing Amount"), 2)        .alias("avg_claim_amount"),
        spark_round(spark_sum("Billing Amount"), 2)  .alias("total_claims"),
        count(when(col("Medical Condition") == "Cancer",       1)) .alias("cancer_patients"),
        count(when(col("Medical Condition") == "Diabetes",     1)) .alias("diabetes_patients"),
        count(when(col("Medical Condition") == "Hypertension", 1)) .alias("hypertension_patients"),
        count(when(col("Medical Condition") == "Arthritis",    1)) .alias("arthritis_patients"),
        count(when(col("Medical Condition") == "Asthma",       1)) .alias("asthma_patients"),
        count(when(col("Medical Condition") == "Obesity",      1)) .alias("obesity_patients")
    )
    .withColumnRenamed("Insurance Provider", "insurance_provider")
    .orderBy(desc("covered_patients"))
    .withColumn("load_timestamp", current_timestamp())
)

logger.info(f"[Table 12] {df_insurance_analysis.count()} rows")
df_insurance_analysis.show(truncate=False)

write_both(df_insurance_analysis,
           table_name  = "insurance_provider_analysis",
           folder_name = "insurance_provider_analysis")


In [0]:
# ============================================================
# PIPELINE SUMMARY
# ============================================================
summary = [
    ("patient_age_group_summary",       df_age_group,              "age_group"),
    ("age_group_condition_summary",     df_age_condition,          "age_group"),
    ("insurance_avg_billing_summary",   df_insurance_billing,      None),
    ("gender_condition_summary",        df_gender_condition,       "gender"),
    ("blood_type_condition_summary",    df_blood_condition,        "blood_type"),
    ("condition_billing_summary",       df_condition_billing,      None),
    ("patient_demographics",            df_patient_demographics,   "gender"),
    ("medical_conditions_summary",      df_conditions,             None),
    ("hospital_performance_metrics",    df_hospital_metrics,       None),
    ("doctor_performance_summary",      df_doctor_metrics,         None),
    ("admission_type_analysis",         df_admission_analysis,     "admission_type"),
    ("insurance_provider_analysis",     df_insurance_analysis,     None),
]

logger.info("=" * 90)
logger.info("PIPELINE COMPLETED SUCCESSFULLY")
logger.info(f"{'Table':<42} {'Rows':>8}  {'Partition':>15}  Destinations")
logger.info("-" * 90)
for table_name, df_ref, partition in summary:
    part_str = partition if partition else "-"
    logger.info(f"  {table_name:<40} {df_ref.count():>8,}  {part_str:>15}  SQL + Gold (Parquet)")
logger.info("=" * 90)
logger.info(f"Gold base path : {ADLS_GOLD_BASE}")
logger.info(f"SQL database   : {SQL_DATABASE} on {SQL_SERVER}")
